# **MIS 545 Project Code**

### Import Packages

In [1]:
!pip install ucimlrepo

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

### Read dataset into a Pandas dataframe

In [3]:
# Read the UCI dataset and create dataframes
url_dataset = fetch_ucirepo(id=967)

# Assign y first, as its column name is needed to derive X if url_dataset.features is None
y = url_dataset.data.targets

# Try to get features directly
X = url_dataset.features

# If url_dataset.features is None or empty, derive X from url_dataset.data.original
if X is None or X.empty:
    if url_dataset.data.original is not None:
        # Get the name of the target column from the 'y' DataFrame (kernel state shows 'label')
        target_column_name = y.columns[0] if not y.empty else 'label'

        if target_column_name and target_column_name in url_dataset.data.original.columns:
            X = url_dataset.data.original.drop(columns=[target_column_name])
        else:
            print(f"Warning: Target column '{target_column_name}' not found in original data. Attempting to use all original data as features.")
            X = url_dataset.data.original.copy()
            # If target_column_name is still in X, drop it as a fallback
            if target_column_name and target_column_name in X.columns:
                X = X.drop(columns=[target_column_name])
    else:
        print("Error: url_dataset.data.original is None. Cannot construct X.")
        X = None

# create a dataframe with X and y
url_dataset = pd.concat([X, y], axis=1)

# Label 1 = legitimate; Label 0 = phishing
#url_dataset.info()

In [4]:
# YOU DO NOT NEED TO USE THIS UNLESS THE CODE ABOVE DOESN'T RUN
# back up code for using dataset until security certificate is reinstated

from google.colab import drive
drive.mount('/content/drive')

url_dataset = pd.read_csv('/content/drive/MyDrive/PhiUSIIL_Phishing_URL_Dataset.csv')

MessageError: Error: credential propagation was unsuccessful

In [5]:
# Keep only URL features (drop HTML features)
url_dataset = url_dataset.drop(columns=['FILENAME', 'URL', 'Domain', 'URLSimilarityIndex', 'Title', 'Robots', 'IsResponsive', 'HasDescription',
                                        'HasExternalFormSubmit','HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank',
                                        'Pay', 'Crypto','HasCopyrightInfo', 'HasTitle', 'HasFavicon', 'LineOfCode', 'LargestLineLength',
                                        'DomainTitleMatchScore', 'URLTitleMatchScore', 'NoOfURLRedirect', 'NoOfSelfRedirect','NoOfPopup',
                                        'NoOfiFrame', 'NoOfImage','NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef'])

url_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235795 entries, 0 to 235794
Data columns (total 23 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   URLLength                   235795 non-null  int64  
 1   DomainLength                235795 non-null  int64  
 2   IsDomainIP                  235795 non-null  int64  
 3   TLD                         235795 non-null  object 
 4   CharContinuationRate        235795 non-null  float64
 5   TLDLegitimateProb           235795 non-null  float64
 6   URLCharProb                 235795 non-null  float64
 7   TLDLength                   235795 non-null  int64  
 8   NoOfSubDomain               235795 non-null  int64  
 9   HasObfuscation              235795 non-null  int64  
 10  NoOfObfuscatedChar          235795 non-null  int64  
 11  ObfuscationRatio            235795 non-null  float64
 12  NoOfLettersInURL            235795 non-null  int64  
 13  LetterRatioInU

### Feature Engineering and SMOTE oversampling

In [6]:
from sklearn.model_selection import train_test_split

# Define X and y and reset index
X = url_dataset.drop(columns=['label'])
y = url_dataset['label']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1)

In [7]:
# Transform TLD column to be is_common_TLD

# Identify common TLDs based on a threshold (e.g., TLDs with at least 1000 occurrences)
# Use X_train for fitting to avoid data leakage
tld_counts_train = X_train['TLD'].value_counts()
common_tlds = tld_counts_train[tld_counts_train >= 1000].index.tolist()

# Create the new binary feature 'is_common_TLD' for X_train and X_test
X_train['is_common_TLD'] = X_train['TLD'].apply(lambda x: 1 if x in common_tlds else 0)
X_test['is_common_TLD'] = X_test['TLD'].apply(lambda x: 1 if x in common_tlds else 0)


# Drop 'TLD' from both X_train and X_test
X_train = X_train.drop(columns=['TLD'])
X_test = X_test.drop(columns=['TLD'])

In [8]:
# see common TLDs
print(common_tlds)

['com', 'org', 'net', 'app', 'uk', 'co', 'io', 'de', 'ru', 'au', 'top', 'dev', 'jp', 'it', 'fr', 'edu', 'br', 'nl', 'ca', 'info', 'link', 'site', 'xyz', 'in', 'pl', 'gov']


In [9]:
# Amount of legit and phishing URLs
# 1 = legit; 0 = phish
url_dataset['label'].value_counts()

,count
label,
1,134850
0,100945


In [10]:
!pip install imblearn

In [11]:
from imblearn.over_sampling import SMOTE
from collections import Counter

print("Original training set class distribution:", Counter(y_train))

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Resampled training set class distribution:", Counter(y_train_resampled))

Original training set class distribution: Counter({1: 107880, 0: 80756})
Resampled training set class distribution: Counter({1: 107880, 0: 107880})


In [12]:
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

#print("Original training set class distribution:", Counter(y_train))

# Apply RandomUnderSampler
#rus = RandomUnderSampler(random_state=42)
#X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

#print("Resampled training set class distribution:", Counter(y_train_resampled))

### Pipeline Creation

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Define the numerical, binary, and categorical features
numeric_features = ['URLLength', 'DomainLength', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength',
                    'NoOfSubDomain', 'SpacialCharRatioInURL','NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL',
                    'NoOfDegitsInURL','DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL','NoOfOtherSpecialCharsInURL']
binary_features = ['IsDomainIP', 'IsHTTPS', 'is_common_TLD', 'HasObfuscation']

# transform columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('bin', 'passthrough', binary_features)
    ],
    remainder='drop'
)

# Models:

### Logistic Regression

In [14]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# RFECV pipeline
rfecv_estimator = LogisticRegression(max_iter=1000, random_state=42)

rfecv_pipeline = Pipeline([
    ('pre', preprocessor),
    ('sel', RFECV(
        estimator=rfecv_estimator,
        cv=5,
        scoring='f1',
        step=1
    ))
])

# Fit feature selector using resampled data
rfecv_pipeline.fit(X_train_resampled, y_train_resampled)

# Extract selected features mask
selector = rfecv_pipeline.named_steps['sel']
selected_mask = selector.support_

# Transform datasets using resampled data for training
X_train_selected = selector.transform(preprocessor.fit_transform(X_train_resampled))
X_test_selected  = selector.transform(preprocessor.transform(X_test))

print("Number of selected features:", selector.n_features_)
print("Selected features:", X_train.columns[selected_mask])

Number of selected features: 6
Selected features: Index(['URLLength', 'DomainLength', 'ObfuscationRatio', 'LetterRatioInURL',
       'NoOfAmpersandInURL', 'SpacialCharRatioInURL'],
      dtype='object')


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

# Model for tuning
lr = LogisticRegression(max_iter=1000, random_state=42)

# Hyperparameter grid
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
}

# GridSearchCV
grid_search = GridSearchCV(
    estimator=lr,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train_selected, y_train_resampled)

best_lr = grid_search.best_estimator_
print("Best hyperparameters:", grid_search.best_params_)

# Evaluate on test
y_pred = best_lr.predict(X_test_selected)
y_prob = best_lr.predict_proba(X_test_selected)

print("\nLogistic Regression Results:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("\nAUROC:", roc_auc_score(y_test, y_prob[:, 1]))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

### Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# RFECV pipeline
rfecv_estimator = DecisionTreeClassifier(criterion='gini', random_state=42)

rfecv_pipeline = Pipeline([
    ('pre', preprocessor),
    ('sel', RFECV(
        estimator=rfecv_estimator,
        cv=5,
        scoring='f1',
        step=1
    ))
])

# Fit feature selector using resampled data
rfecv_pipeline.fit(X_train_resampled, y_train_resampled)

# Extract selected features mask
selector = rfecv_pipeline.named_steps['sel']
selected_mask = selector.support_

# Transform datasets using resampled data for training
X_train_selected = selector.transform(preprocessor.fit_transform(X_train_resampled))
X_test_selected  = selector.transform(preprocessor.transform(X_test))

print("Number of selected features:", selector.n_features_)
print("Selected features:", X_train.columns[selected_mask])

Number of selected features: 6
Selected features: Index(['DomainLength', 'TLDLength', 'NoOfLettersInURL', 'NoOfDegitsInURL',
       'NoOfAmpersandInURL', 'SpacialCharRatioInURL'],
      dtype='object')


In [ ]:
# Model for tuning
dt = DecisionTreeClassifier(criterion='gini', random_state=42)

# Hyperparameter grid
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'criterion': ['gini', 'entropy']
}

# GridSearchCV
grid_search = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train_selected, y_train_resampled)

best_dt = grid_search.best_estimator_
print("Best hyperparameters:", grid_search.best_params_)

# Evaluate on test
y_pred = best_dt.predict(X_test_selected)
y_prob = best_dt.predict_proba(X_test_selected)

print("\nDecision Tree Results:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("\nAUROC:", roc_auc_score(y_test, y_prob[:, 1]))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Best hyperparameters: {'criterion': 'gini', 'max_depth': None}

Decision Tree Results:
Accuracy: 0.9976250556627579
Precision: 0.9964143131746267
Recall: 0.9994438264738599
F1-score: 0.9979267705749509

AUROC: 0.9969108048702461

Confusion Matrix:
[[20092    97]
 [   15 26955]]


### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# RFECV pipeline
rfecv_estimator = RandomForestClassifier(n_estimators=20, random_state=42)

rfecv_pipeline = Pipeline([
    ('pre', preprocessor),
    ('sel', RFECV(
        estimator=rfecv_estimator,
        cv=5,
        scoring='f1',
        step=1
    ))
])

# Fit feature selector using resampled data
rfecv_pipeline.fit(X_train_resampled, y_train_resampled)

# Extract selected features mask
selector = rfecv_pipeline.named_steps['sel']
selected_mask = selector.support_

# Transform datasets using resampled data for training
X_train_selected = selector.transform(preprocessor.fit_transform(X_train_resampled))
X_test_selected  = selector.transform(preprocessor.transform(X_test))

print("Number of selected features:", selector.n_features_)
print("Selected features:", X_train.columns[selected_mask])

Number of selected features: 18
Selected features: Index(['URLLength', 'DomainLength', 'IsDomainIP', 'CharContinuationRate',
       'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain',
       'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL',
       'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL',
       'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL',
       'SpacialCharRatioInURL', 'IsHTTPS'],
      dtype='object')


In [ ]:
# Model for tuning
rf = RandomForestClassifier(random_state=42)

# Hyperparameter grid
param_grid = {
    'n_estimators': [10, 20, 50, 75],
    'criterion': ['gini', 'entropy']
}

# GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train_selected, y_train_resampled)

best_dt = grid_search.best_estimator_
print("Best hyperparameters:", grid_search.best_params_)

# Evaluate on test
y_pred = best_dt.predict(X_test_selected)
y_prob = best_dt.predict_proba(X_test_selected)

print("\nRandom Forest Results:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("\nAUROC:", roc_auc_score(y_test, y_prob[:, 1]))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Best hyperparameters: {'criterion': 'entropy', 'n_estimators': 50}

Random Forest Results:
Accuracy: 0.9971161390190632
Precision: 0.9968155224764867
Recall: 0.9981460882461994
F1-score: 0.9974803616422113

AUROC: 0.9988797814674316

Confusion Matrix:
[[20103    86]
 [   50 26920]]


### SVM with Linear kernel

In [ ]:
from sklearn.svm import SVC, LinearSVC

# RFECV pipeline
rfecv_estimator = LinearSVC(max_iter=5000, random_state=42)

rfecv_pipeline = Pipeline([
    ('pre', preprocessor),
    ('sel', RFECV(
        estimator=rfecv_estimator,
        cv=5,
        scoring='f1',
        step=1
    ))
])

# Fit feature selector using resampled data
rfecv_pipeline.fit(X_train_resampled, y_train_resampled)

# Extract selected features mask
selector = rfecv_pipeline.named_steps['sel']
selected_mask = selector.support_

# Transform datasets using resampled data for training
X_train_selected = selector.transform(preprocessor.fit_transform(X_train_resampled))
X_test_selected  = selector.transform(preprocessor.transform(X_test))

print("Number of selected features:", selector.n_features_)
print("Selected features:", X_train.columns[selected_mask])

Number of selected features: 17
Selected features: Index(['URLLength', 'DomainLength', 'TLDLegitimateProb', 'URLCharProb',
       'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar',
       'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL',
       'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL',
       'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL',
       'SpacialCharRatioInURL', 'IsHTTPS'],
      dtype='object')


In [ ]:
# Model for tuning
linear_svm = LinearSVC(max_iter=5000, random_state=42)

# Hyperparameter grid for LinearSVC
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2']
}

# GridSearchCV
grid_search = GridSearchCV(
    estimator=linear_svm,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train_selected, y_train_resampled)

best_linear_svm = grid_search.best_estimator_
print("Best hyperparameters:", grid_search.best_params_)

# Evaluate on test
y_pred = best_linear_svm.predict(X_test_selected)
# LinearSVC does not have predict_proba
# y_prob = best_linear_svm.predict_proba(X_test_selected)

print("\nLinear SVM Results:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("\nAUROC:", roc_auc_score(y_test, y_prob[:, 1]))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Best hyperparameters: {'C': 10, 'penalty': 'l2'}

Linear SVM Results:
Accuracy: 0.9977734896838355
Precision: 0.9961585343330994
Recall: 0.999962921764924
F1-score: 0.9980571026774976

AUROC: 0.9976047026713613

Confusion Matrix:
[[20085   104]
 [    1 26969]]


### KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SelectFromModel

# Use LogisticRegression with L1 penalty (Lasso) as the feature selection model
lasso_selector_model = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    random_state=42,
    C=0.1
)

# Create a pipeline
feature_selection_pipeline = Pipeline([
    ('pre', preprocessor),
    ('sel', SelectFromModel(estimator=lasso_selector_model, prefit=False))
])

# Fit feature selector using resampled data
feature_selection_pipeline.fit(X_train_resampled, y_train_resampled)

# Extract selected features mask
selector = feature_selection_pipeline.named_steps['sel']
selected_mask = selector.get_support()

# Transform datasets using resampled data for training
X_train_selected = feature_selection_pipeline.transform(X_train_resampled)
X_test_selected = feature_selection_pipeline.transform(X_test)

# print the number of features selected
print("Number of selected features:", selected_mask.sum())
print("Selected features:", X_train.columns[selected_mask])

Number of selected features: 14
Selected features: Index(['DomainLength', 'IsDomainIP', 'CharContinuationRate',
       'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain',
       'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL',
       'NoOfDegitsInURL', 'NoOfAmpersandInURL', 'SpacialCharRatioInURL',
       'IsHTTPS'],
      dtype='object')


In [ ]:
# Model for tuning
knn = KNeighborsClassifier(p=2)

# Hyperparameter grid
param_grid = {
    'n_neighbors': [3, 5, 7, 10],
    'p': [1, 2]
}

# GridSearchCV
grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train_selected, y_train_resampled)

best_knn = grid_search.best_estimator_
print("Best hyperparameters:", grid_search.best_params_)

# Evaluate on test
y_pred = best_knn.predict(X_test_selected)
y_prob = best_knn.predict_proba(X_test_selected)

print("\nKNN Results:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("\nAUROC:", roc_auc_score(y_test, y_prob[:, 1]))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Best hyperparameters: {'n_neighbors': 3, 'p': 2}

KNN Results:
Accuracy: 0.9966708369558303
Precision: 0.9955276289040843
Recall: 0.9986651835372636
F1-score: 0.9970939379916706

AUROC: 0.9976047026713613

Confusion Matrix:
[[20068   121]
 [   36 26934]]
